# Demographic, Clinical, and Laboratory Measurements from HIV Patients on TLD at Machakos Level 5 Hospital, Kenya (2024) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.s85d-x5d7) dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset is described in Croissant schema and is accessible from the following URL:

`https://sen.science/doi/10.71728/senscience.s85d-x5d7/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.s85d-x5d7/fair2.json"

# Load the Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
# Print dataset summary
print("Name:", metadata.name)
print("Version:", metadata.version)
print("Identifier:", getattr(metadata, 'identifier', 'N/A'))
print("Published date:", getattr(metadata, 'datePublished', 'N/A'))
print("Description:\n", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs. We explore the dataset's structure via its Croissant metadata. All Croissant entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.recordSets
print(f"Total record sets found: {len(record_sets)}\n")

recordset_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id_}")
    recordset_ids.append(rs.id_)
    if hasattr(rs, 'fields'):
        print(f"  Fields [@id] and [name]:")
        for f in rs.fields:
            print(f"    - {f.id_} [{f.name}]")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. This dataset contains clinical and demographic measurements as described above. All columns/fields are referenced via their `@id` for programmatic access.

In [ ]:
# Extract all record sets into DataFrames using @id

dfs = {}
for rs_id in recordset_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dfs[rs_id] = df
    print(f"Loaded RecordSet '@id': {rs_id} with shape {df.shape}")

# Preview the first record set
primary_rs_id = recordset_ids[0]
print(f"\nAll columns in first RecordSet '@id': {primary_rs_id}")
print(dfs[primary_rs_id].columns.tolist())
dfs[primary_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform basic data processing using field `@id`s. 
- We'll select a numeric field (e.g., serum creatinine or patient age), filter on it, and normalize its values.
- We'll also group by a categorical field (e.g., sex/gender or kidney disease outcome).

**Note:** Be sure to replace the `field_id`s with the ones relevant to your use (using the output from above if changing to a different field).

In [ ]:
# Select RecordSet to analyze
record_set_id = primary_rs_id
df = dfs[record_set_id]

# Choose field @id for numeric and group fields (update if needed based on overview)
# Example field @id (these may need to be changed based on your dataset's actual output above):
# E.g., 'cr:age', 'cr:serum_creatinine', 'cr:sex', 'cr:kidney_disease_outcome'
numeric_field_id = None
group_field_id = None

# Guess likely field @id for demonstration
for c in df.columns:
    if 'creatinine' in c.lower():
        numeric_field_id = c
    elif 'age' == c.lower() and numeric_field_id is None:
        numeric_field_id = c
    if group_field_id is None and ('sex' in c.lower() or 'gender' in c.lower()):
        group_field_id = c
    if group_field_id is None and ('kidney_disease' in c or 'outcome' in c):
        group_field_id = c

print(f"Numeric field for EDA: {numeric_field_id}")
print(f"Group field for EDA: {group_field_id}")

# Drop records where numeric field is missing
filtered_df = df.dropna(subset=[numeric_field_id])

# Filter for more meaningful records (e.g., nonzero/positive values)
if filtered_df[numeric_field_id].dtype != 'O':
    filtered_df = filtered_df[filtered_df[numeric_field_id] > 0]

# Normalize the numeric field
filtered_df[numeric_field_id + '_normalized'] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"First 5 values after normalization:")
print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

if group_field_id in filtered_df.columns:
    # Show mean of numeric field by group
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} by {group_field_id}:")
    print(grouped)

## 5. Visualization
Let's plot the distribution of our selected numeric field, and (if present) a boxplot by a grouping variable (e.g., sex or outcome).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Plot histogram
plt.figure(figsize=(8, 4))
sns.histplot(data=filtered_df, x=numeric_field_id, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field is available, show boxplot
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 dataset using `mlcroissant`, referencing all data schema entities by their `@id` for traceable, standards-based analysis. This process enables transparent data handling, compatible with FAIR and Croissant best practices. Further analysis can build on this pipeline for risk factor modeling or epidemiologic reporting.

<!-- End of notebook -->